Run this notebook in a clean Kaggle session with a Tesla P100 GPU and the repository plus AI4Mars dataset attached. Run the dependency cell before any torch import; do not install `requirements.txt`. The CUDA preflight must pass and the smoke configuration should complete before starting the full run.

In [ ]:
# 1. Fetch the project
!git clone --depth 1 --branch main https://github.com/Jacob-Miller-s/AI4Mars.git /kaggle/working/AI4Mars

In [ ]:
# 2. Install the P100-compatible dependencies
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/AI4Mars")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "-r",
    str(PROJECT_ROOT / "requirements-kaggle.txt"),
])

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/AI4Mars")
sys.path.insert(0, str(PROJECT_ROOT))

from ai4mars import run_full_reproduction
from ai4mars.cuda import run_cuda_preflight

run_cuda_preflight()

matches = list(Path("/kaggle/input").glob("**/ai4mars-dataset-merged-0.6"))
if len(matches) != 1:
    raise RuntimeError("Attach the extracted AI4Mars merged 0.6 dataset.")
    
run_dir = run_full_reproduction(
    config_path=PROJECT_ROOT / "configs/reproduction/paper_deeplabv3plus_kaggle_p100.yaml",
    dataset_root=Path(os.environ.get("AI4MARS_DATASET_ROOT", matches[0])),
    manifest_root=PROJECT_ROOT / "artifacts/manifests",
    output_root=Path("/kaggle/working/ai4mars-paper-reproduction"),
    run_id=f"deeplabv3plus-tesla-p100-{datetime.now():%Y%m%d-%H%M%S}",
    resume_checkpoint=Path(os.environ["AI4MARS_RESUME_CHECKPOINT"])
        if os.environ.get("AI4MARS_RESUME_CHECKPOINT")
        else None,
    device="cuda",
)

print(f"Completed run: {run_dir}")

In [ ]:
import json

import matplotlib.pyplot as plt

epochs = [json.loads(line) for line in (run_dir / 'metrics.jsonl').read_text(encoding='utf-8').splitlines()]
epoch_numbers = [item['epoch'] for item in epochs]
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epoch_numbers, [item['train_loss'] for item in epochs], label='Train')
axes[0].plot(epoch_numbers, [item['val_loss'] for item in epochs], label='Validation')
axes[0].set(title='Loss by completed epoch', xlabel='Epoch', ylabel='Cross-entropy')
axes[0].legend()
axes[1].plot(epoch_numbers, [item['mean_iou'] for item in epochs], color='#047857')
axes[1].set(title='Validation mIoU by completed epoch', xlabel='Epoch', ylabel='mIoU', ylim=(0, 1))
figure.tight_layout()
plt.show()